# 4049. Count Values With Equally Spaced Occurrences II — Nhập môn Hash Map gom nhóm chỉ số

Bài toán LeetCode (Medium):

```python
class Solution:
    def countSpecialIntegers(self, nums: list[int]) -> int:
        ...
```

## Mục tiêu của notebook

Sau notebook này, bạn sẽ hiểu:

- "Các lần xuất hiện **cách đều nhau**" nghĩa là gì — và đó chính là **cấp số cộng**.
- Lời giải Brute Force: với mỗi giá trị, quét lại cả mảng — vì sao chậm.
- **Hash Map gom nhóm**: `dict` ánh xạ giá trị → **danh sách chỉ số**, dùng `defaultdict(list)`.
- Kiểm tra khoảng cách liên tiếp bằng `zip(idx, idx[1:])`, `all(...)`, `set(...)`.
- Mẹo Pythonic: so sánh với `range(first, last + 1, gap)`.
- Lời giải **một lượt duyệt, bộ nhớ nhỏ**: mỗi giá trị chỉ lưu 4 thông tin.
- Trace, code chuẩn LeetCode, lỗi hay mắc, biến thể và bài tập có đáp án.

# 1. Đề bài nói gì?

Cho mảng số nguyên `nums`. Một số nguyên `x` được gọi là **special** nếu:

1. `x` xuất hiện **ít nhất 3 lần** trong `nums`.
2. Mọi lần xuất hiện của `x` **cách đều nhau**: nếu các vị trí là `i1 < i2 < ... < im` thì
   `i2 - i1 = i3 - i2 = ... = im - i(m-1)`.

Trả về **số lượng giá trị khác nhau** là special.

### Ví dụ

| nums | Kết quả | Giải thích |
|---|---:|---|
| `[1,8,1,5,1,5,8,5]` | 2 | `1` ở `0,2,4` (cách 2); `5` ở `3,5,7` (cách 2); `8` chỉ 2 lần |
| `[8,8,8,8]` | 1 | `8` ở `0,1,2,3` (cách 1) |
| `[8,6,6,8,8]` | 0 | `8` ở `0,3,4` (cách 3 rồi 1 — không đều); `6` chỉ 2 lần |

### Ràng buộc

- `3 <= n <= 10^5`
- `1 <= nums[i] <= 10^9`

# 2. "Cách đều" = cấp số cộng

Dãy vị trí `i1, i2, ..., im` cách đều nhau khi **mọi hiệu liên tiếp bằng nhau**. Dãy như vậy gọi là **cấp số cộng** với **công sai** `d = i2 - i1`.

Ví dụ với `nums = [1, 8, 1, 5, 1, 5, 8, 5]`:

```text
chỉ số:  0  1  2  3  4  5  6  7
nums:    1  8  1  5  1  5  8  5

giá trị 1: vị trí 0, 2, 4      hiệu: 2, 2      -> đều ✔ (≥ 3 lần)
giá trị 5: vị trí 3, 5, 7      hiệu: 2, 2      -> đều ✔ (≥ 3 lần)
giá trị 8: vị trí 1, 6         chỉ 2 lần       -> ✘
```

Ví dụ `[8, 6, 6, 8, 8]`:

```text
giá trị 8: vị trí 0, 3, 4      hiệu: 3, 1      -> không đều ✘
```

Vậy công việc gồm hai bước:

1. **Gom** các vị trí của từng giá trị.
2. **Kiểm tra** mỗi nhóm: có `>= 3` phần tử và mọi hiệu liên tiếp bằng nhau.

# 3. Cách 1 — Brute Force: với mỗi giá trị, quét lại cả mảng

- `set(nums)` cho các giá trị **khác nhau**.
- Với mỗi giá trị `x`, duyệt cả mảng để lấy các vị trí của `x`.
- Kiểm tra các hiệu.

In [1]:
def is_equally_spaced(positions):
    """True nếu mọi hiệu liên tiếp bằng nhau (positions đã tăng dần)."""
    gap = positions[1] - positions[0]
    for k in range(2, len(positions)):
        if positions[k] - positions[k - 1] != gap:
            return False
    return True


def count_special_bruteforce(nums):
    count = 0
    for x in set(nums):
        positions = [i for i in range(len(nums)) if nums[i] == x]
        if len(positions) >= 3 and is_equally_spaced(positions):
            count += 1
    return count


print(count_special_bruteforce([1, 8, 1, 5, 1, 5, 8, 5]))   # 2
print(count_special_bruteforce([8, 8, 8, 8]))               # 1
print(count_special_bruteforce([8, 6, 6, 8, 8]))            # 0

2
1
0


Chú ý thứ tự trong điều kiện `len(positions) >= 3 and is_equally_spaced(positions)`: Python **dừng sớm** ở `and` — nếu vế trái `False` thì không tính vế phải. Nhờ vậy `is_equally_spaced` không bao giờ bị gọi với list có ít hơn 2 phần tử (sẽ lỗi `IndexError` ở `positions[1]`).

## Độ phức tạp của Brute Force

Gọi `d` là số giá trị khác nhau. Mỗi giá trị quét cả mảng `O(n)` → tổng `O(n · d)`. Khi mọi phần tử khác nhau (`d = n = 10^5`) → `10^10` bước. Quá chậm!

**Nút thắt:** ta đọc lại cả mảng cho **mỗi** giá trị. Lẽ ra chỉ cần đọc mảng **một lần**, và mỗi phần tử tự "báo danh" vào nhóm của nó.

# 4. Nền tảng Python — `dict` gom nhóm và `defaultdict`

Ý tưởng: dùng `dict` với **key = giá trị**, **value = list các vị trí**. Duyệt mảng một lần, gặp `nums[i] = x` thì thêm `i` vào list của `x`.

Cách viết cơ bản:

```python
groups = {}
for i, x in enumerate(nums):
    if x not in groups:
        groups[x] = []        # lần đầu gặp x: tạo list rỗng
    groups[x].append(i)
```

Cách gọn hơn với `collections.defaultdict(list)`: khi truy cập một key **chưa có**, nó **tự tạo** `list()` rỗng.

In [2]:
nums = [1, 8, 1, 5, 1, 5, 8, 5]

groups = {}
for i, x in enumerate(nums):
    if x not in groups:
        groups[x] = []
    groups[x].append(i)
print("dict thường :", groups)

from collections import defaultdict

groups2 = defaultdict(list)
for i, x in enumerate(nums):
    groups2[x].append(i)          # không cần kiểm tra key
print("defaultdict :", dict(groups2))

dict thường : {1: [0, 2, 4], 8: [1, 6], 5: [3, 5, 7]}
defaultdict : {1: [0, 2, 4], 8: [1, 6], 5: [3, 5, 7]}


Vì ta duyệt `i` **tăng dần**, mỗi list vị trí tự động được **sắp xếp tăng dần** — không cần `sort`.

# 5. Nền tảng Python — Lấy các cặp liên tiếp bằng `zip`

`zip(a, b)` ghép từng cặp phần tử cùng vị trí của hai dãy. Mẹo: `zip(idx, idx[1:])` ghép mỗi phần tử với **phần tử ngay sau nó**.

```text
idx      = [0, 3, 4]
idx[1:]  = [3, 4]
zip      -> (0, 3), (3, 4)       -> hiệu: 3, 1
```

Hai hàm tiện dụng:

- `all(điều_kiện for ...)`: `True` nếu **mọi** phần tử thoả.
- `set(...)`: loại trùng — "mọi hiệu bằng nhau" ⇔ **tập các hiệu có đúng 1 phần tử**.

In [3]:
idx = [0, 3, 4]
print(list(zip(idx, idx[1:])))
gaps = [b - a for a, b in zip(idx, idx[1:])]
print("các hiệu:", gaps, "| tập các hiệu:", set(gaps))

idx = [3, 5, 7]
gaps = [b - a for a, b in zip(idx, idx[1:])]
print("các hiệu:", gaps, "| tập các hiệu:", set(gaps), "| đều nhau?", len(set(gaps)) == 1)
print("all(...):", all(g == gaps[0] for g in gaps))

[(0, 3), (3, 4)]
các hiệu: [3, 1] | tập các hiệu: {1, 3}
các hiệu: [2, 2] | tập các hiệu: {2} | đều nhau? True
all(...): True


# 6. Cách 2 — Hash Map gom nhóm + kiểm tra từng nhóm

```text
1. groups = defaultdict(list); duyệt mảng một lần, gom vị trí.
2. Với mỗi list vị trí idx:
       nếu len(idx) >= 3 và len(set(các hiệu)) == 1  -> special
```

In [4]:
def count_special_groups(nums):
    groups = defaultdict(list)
    for i, x in enumerate(nums):
        groups[x].append(i)

    count = 0
    for x, idx in groups.items():
        if len(idx) < 3:
            continue
        gaps = {b - a for a, b in zip(idx, idx[1:])}    # set comprehension
        if len(gaps) == 1:
            count += 1
    return count


print(count_special_groups([1, 8, 1, 5, 1, 5, 8, 5]))   # 2
print(count_special_groups([8, 8, 8, 8]))               # 1
print(count_special_groups([8, 6, 6, 8, 8]))            # 0

2
1
0


`{b - a for a, b in ...}` là **set comprehension**: giống list comprehension nhưng dùng `{}` và kết quả là `set` (tự loại trùng).

`groups.items()` duyệt cả **key và value** của dict cùng lúc.

## Mô phỏng cách 2

`nums = [1, 8, 1, 5, 1, 5, 8, 5]`

**Bước gom nhóm:**

| i | nums[i] | groups sau bước này |
|---|---|---|
| 0 | 1 | `{1: [0]}` |
| 1 | 8 | `{1: [0], 8: [1]}` |
| 2 | 1 | `{1: [0, 2], 8: [1]}` |
| 3 | 5 | `{1: [0, 2], 8: [1], 5: [3]}` |
| 4 | 1 | `{1: [0, 2, 4], 8: [1], 5: [3]}` |
| 5 | 5 | `{1: [0, 2, 4], 8: [1], 5: [3, 5]}` |
| 6 | 8 | `{1: [0, 2, 4], 8: [1, 6], 5: [3, 5]}` |
| 7 | 5 | `{1: [0, 2, 4], 8: [1, 6], 5: [3, 5, 7]}` |

**Bước kiểm tra:**

| giá trị | vị trí | số lần | tập hiệu | special? |
|---|---|---:|---|---|
| 1 | `[0, 2, 4]` | 3 | `{2}` | ✔ |
| 8 | `[1, 6]` | 2 | — | ✘ (ít hơn 3) |
| 5 | `[3, 5, 7]` | 3 | `{2}` | ✔ |

In [5]:
def trace_groups(nums):
    groups = defaultdict(list)
    for i, x in enumerate(nums):
        groups[x].append(i)
        print(f"i={i}, x={x}: groups = {dict(groups)}")
    print("-" * 60)
    count = 0
    for x, idx in groups.items():
        if len(idx) < 3:
            print(f"giá trị {x}: vị trí {idx} -> chỉ {len(idx)} lần, loại")
            continue
        gaps = {b - a for a, b in zip(idx, idx[1:])}
        ok = len(gaps) == 1
        count += ok
        print(f"giá trị {x}: vị trí {idx}, tập hiệu {gaps} -> {'special' if ok else 'không đều'}")
    print("Đáp án:", count)
    return count

trace_groups([1, 8, 1, 5, 1, 5, 8, 5])
print()
trace_groups([8, 6, 6, 8, 8])

i=0, x=1: groups = {1: [0]}
i=1, x=8: groups = {1: [0], 8: [1]}
i=2, x=1: groups = {1: [0, 2], 8: [1]}
i=3, x=5: groups = {1: [0, 2], 8: [1], 5: [3]}
i=4, x=1: groups = {1: [0, 2, 4], 8: [1], 5: [3]}
i=5, x=5: groups = {1: [0, 2, 4], 8: [1], 5: [3, 5]}
i=6, x=8: groups = {1: [0, 2, 4], 8: [1, 6], 5: [3, 5]}
i=7, x=5: groups = {1: [0, 2, 4], 8: [1, 6], 5: [3, 5, 7]}
------------------------------------------------------------
giá trị 1: vị trí [0, 2, 4], tập hiệu {2} -> special
giá trị 8: vị trí [1, 6] -> chỉ 2 lần, loại
giá trị 5: vị trí [3, 5, 7], tập hiệu {2} -> special
Đáp án: 2

i=0, x=8: groups = {8: [0]}
i=1, x=6: groups = {8: [0], 6: [1]}
i=2, x=6: groups = {8: [0], 6: [1, 2]}
i=3, x=8: groups = {8: [0, 3], 6: [1, 2]}
i=4, x=8: groups = {8: [0, 3, 4], 6: [1, 2]}
------------------------------------------------------------
giá trị 8: vị trí [0, 3, 4], tập hiệu {1, 3} -> không đều
giá trị 6: vị trí [1, 2] -> chỉ 2 lần, loại
Đáp án: 0


0

Dòng `count += ok`: trong Python `True` tương đương `1`, `False` tương đương `0`, nên cộng một giá trị `bool` vào số nguyên là hợp lệ.

# 7. Cách 3 — Mẹo Pythonic: so với `range`

Một cấp số cộng bắt đầu ở `first`, công sai `gap`, kết thúc ở `last` chính là `range(first, last + 1, gap)`.

Vậy: `idx` cách đều ⇔ `idx == list(range(idx[0], idx[-1] + 1, idx[1] - idx[0]))`.

In [6]:
idx = [3, 5, 7]
print(list(range(idx[0], idx[-1] + 1, idx[1] - idx[0])))      # [3, 5, 7] -> bằng idx
idx = [0, 3, 4]
print(list(range(idx[0], idx[-1] + 1, idx[1] - idx[0])))      # [0, 3] -> khác idx


def count_special_range_trick(nums):
    groups = defaultdict(list)
    for i, x in enumerate(nums):
        groups[x].append(i)
    count = 0
    for idx in groups.values():
        if len(idx) >= 3 and idx == list(range(idx[0], idx[-1] + 1, idx[1] - idx[0])):
            count += 1
    return count

print(count_special_range_trick([1, 8, 1, 5, 1, 5, 8, 5]))   # 2
print(count_special_range_trick([8, 6, 6, 8, 8]))            # 0

[3, 5, 7]
[0, 3]
2
0


Vì sao an toàn khi `gap = idx[1] - idx[0]`? Các vị trí khác nhau và tăng dần nên `gap >= 1` — `range` với bước dương, không bao giờ bước `0` (sẽ lỗi `ValueError`).

### Cẩn thận: mẹo gọn nhưng có bẫy hiệu năng

Cách này ngắn, nhưng tạo thêm một list tạm cho mỗi nhóm. Độ dài list `range` là khoảng `(last - first) / gap + 1` — **không** nhất thiết bằng `len(idx)`.

Nếu nhóm **không** đều, list `range` có thể **dài hơn rất nhiều**: ví dụ `idx = [0, 1, 99999]` cho `range(0, 100000, 1)` dài `10^5` dù nhóm chỉ có 3 phần tử. Nhiều nhóm như vậy sẽ làm tổng thời gian vượt xa `O(n)`.

Cách sửa: kiểm tra độ dài trước, chỉ tạo `range` khi độ dài khớp:

```python
expected_len = (idx[-1] - idx[0]) // gap + 1
if expected_len == len(idx) and idx == list(range(...)):
```

Đây là bài học về việc **mẹo gọn chưa chắc an toàn** — cần nghĩ về trường hợp xấu.

# 8. Cách 4 — Một lượt duyệt, mỗi giá trị chỉ lưu 4 thông tin

Cách 2 lưu **toàn bộ** list vị trí. Thật ra để kiểm tra "cách đều", ta chỉ cần nhớ cho mỗi giá trị:

| Thông tin | Ý nghĩa |
|---|---|
| `last` | vị trí lần xuất hiện gần nhất |
| `gap` | khoảng cách chuẩn (hiệu đầu tiên), `None` nếu mới gặp 1 lần |
| `count` | số lần xuất hiện |
| `ok` | đến giờ vẫn cách đều? |

Khi gặp lại `x` tại vị trí `i`:

```text
g = i - last
nếu gap là None (đây là lần thứ 2): gap = g
ngược lại nếu g != gap: ok = False       (hỏng vĩnh viễn)
last = i ; count += 1
```

Cuối cùng đếm các giá trị có `count >= 3` và `ok`.

Đây giống cách **kiểm tra trước, cập nhật sau** trong bài Two Sum: luôn so sánh với thông tin **cũ** rồi mới ghi đè.

In [7]:
def count_special_one_pass(nums):
    state = {}                       # x -> [last, gap, count, ok]
    for i, x in enumerate(nums):
        if x not in state:
            state[x] = [i, None, 1, True]
            continue
        s = state[x]
        g = i - s[0]
        if s[1] is None:
            s[1] = g                 # lần thứ 2: xác định khoảng cách chuẩn
        elif g != s[1]:
            s[3] = False             # lệch khoảng cách -> không bao giờ special
        s[0] = i
        s[2] += 1
    return sum(1 for last, gap, cnt, ok in state.values() if cnt >= 3 and ok)


print(count_special_one_pass([1, 8, 1, 5, 1, 5, 8, 5]))   # 2
print(count_special_one_pass([8, 8, 8, 8]))               # 1
print(count_special_one_pass([8, 6, 6, 8, 8]))            # 0

2
1
0


Hai điểm Python đáng chú ý:

- `s = state[x]` rồi sửa `s[0]`, `s[1]`... sẽ **sửa luôn list trong dict** (cùng một đối tượng — aliasing, lần này có ích!).
- `is None` là cách chuẩn để kiểm tra `None` (thay vì `== None`). Không nên viết `if not s[1]`: biểu thức đó cũng đúng khi `s[1] == 0`. Ở bài này `gap` không bao giờ bằng `0`, nhưng ở bài khác `0` có thể là giá trị hợp lệ — thói quen dùng `is None` giúp tránh lỗi khó thấy.

## Trace cách 4 với `[8, 6, 6, 8, 8]`

| i | x | trạng thái của x trước | g = i − last | hành động | trạng thái của x sau |
|---|---|---|---:|---|---|
| 0 | 8 | — | — | tạo mới | `[0, None, 1, True]` |
| 1 | 6 | — | — | tạo mới | `[1, None, 1, True]` |
| 2 | 6 | `[1, None, 1, True]` | 1 | gap = 1 | `[2, 1, 2, True]` |
| 3 | 8 | `[0, None, 1, True]` | 3 | gap = 3 | `[3, 3, 2, True]` |
| 4 | 8 | `[3, 3, 2, True]` | 1 | 1 ≠ 3 → ok = False | `[4, 3, 3, False]` |

Kết thúc: `8` có `count = 3` nhưng `ok = False`; `6` có `count = 2`. Đáp án `0` ✔.

In [8]:
def trace_one_pass(nums):
    state = {}
    for i, x in enumerate(nums):
        before = list(state[x]) if x in state else None
        if x not in state:
            state[x] = [i, None, 1, True]
            note = "tạo mới"
        else:
            s = state[x]
            g = i - s[0]
            if s[1] is None:
                s[1] = g
                note = f"g={g}: đặt gap = {g}"
            elif g != s[1]:
                s[3] = False
                note = f"g={g} khác gap={s[1]} -> ok = False"
            else:
                note = f"g={g} khớp gap"
            s[0] = i
            s[2] += 1
        print(f"i={i}, x={x}: trước {before} | {note} | sau {state[x]}")
    ans = sum(1 for last, gap, cnt, ok in state.values() if cnt >= 3 and ok)
    print("Đáp án:", ans)
    return ans

trace_one_pass([8, 6, 6, 8, 8])

i=0, x=8: trước None | tạo mới | sau [0, None, 1, True]
i=1, x=6: trước None | tạo mới | sau [1, None, 1, True]
i=2, x=6: trước [1, None, 1, True] | g=1: đặt gap = 1 | sau [2, 1, 2, True]
i=3, x=8: trước [0, None, 1, True] | g=3: đặt gap = 3 | sau [3, 3, 2, True]
i=4, x=8: trước [3, 3, 2, True] | g=1 khác gap=3 -> ok = False | sau [4, 3, 3, False]
Đáp án: 0


0

# 9. Lời giải LeetCode chuẩn

Ta chọn **cách 2** (gom nhóm) vì rõ ràng nhất; cách 4 cũng rất tốt nếu muốn tiết kiệm bộ nhớ.

In [9]:
from collections import defaultdict

class Solution:
    def countSpecialIntegers(self, nums: list[int]) -> int:
        groups = defaultdict(list)
        for i, x in enumerate(nums):
            groups[x].append(i)

        count = 0
        for idx in groups.values():
            if len(idx) < 3:
                continue
            gap = idx[1] - idx[0]
            if all(b - a == gap for a, b in zip(idx, idx[1:])):
                count += 1
        return count

In [10]:
sol = Solution()

tests = [
    ([1, 8, 1, 5, 1, 5, 8, 5], 2),
    ([8, 8, 8, 8], 1),
    ([8, 6, 6, 8, 8], 0),
    ([1, 2, 3], 0),                          # mọi giá trị chỉ 1 lần
    ([7, 7, 7], 1),
    ([7, 0, 7, 7], 0),                       # vị trí 0, 2, 3
    ([4, 9, 4, 9, 4, 9, 4], 2),              # 4: 0,2,4,6 ; 9: 1,3,5
    ([5, 1, 1, 5, 1, 1, 5], 1),              # 5: 0,3,6 đều; 1: 1,2,4,5 không đều
    ([10**9] * 5, 1),
]
for nums, expected in tests:
    result = sol.countSpecialIntegers(nums)
    print(f"{str(nums):38} -> {result} | expected {expected} | {'PASS' if result == expected else 'FAIL'}")
    assert result == expected

[1, 8, 1, 5, 1, 5, 8, 5]               -> 2 | expected 2 | PASS
[8, 8, 8, 8]                           -> 1 | expected 1 | PASS
[8, 6, 6, 8, 8]                        -> 0 | expected 0 | PASS
[1, 2, 3]                              -> 0 | expected 0 | PASS
[7, 7, 7]                              -> 1 | expected 1 | PASS
[7, 0, 7, 7]                           -> 0 | expected 0 | PASS
[4, 9, 4, 9, 4, 9, 4]                  -> 2 | expected 2 | PASS
[5, 1, 1, 5, 1, 1, 5]                  -> 1 | expected 1 | PASS
[1000000000, 1000000000, 1000000000, 1000000000, 1000000000] -> 1 | expected 1 | PASS


In [11]:
import random, time

random.seed(49)
for _ in range(3000):
    nums = [random.randint(1, 3) for _ in range(random.randint(3, 12))]
    b = count_special_bruteforce(nums)
    assert count_special_groups(nums) == b
    assert count_special_range_trick(nums) == b
    assert count_special_one_pass(nums) == b
    assert sol.countSpecialIntegers(nums) == b
print("Mọi cách khớp nhau trên 3000 mảng ngẫu nhiên ✔")

big = [random.randint(1, 10**9) for _ in range(10**5)]
for f in (count_special_groups, count_special_one_pass, sol.countSpecialIntegers):
    t0 = time.time()
    f(big)
    print(f"{f.__name__:24}: {time.time() - t0:.3f} giây")

Mọi cách khớp nhau trên 3000 mảng ngẫu nhiên ✔
count_special_groups    : 0.050 giây


count_special_one_pass  : 0.068 giây


countSpecialIntegers    : 0.027 giây


# 10. Vì sao thuật toán đúng?

- **Gom nhóm đúng:** mỗi chỉ số `i` được thêm vào đúng một list — list của `nums[i]`. Do duyệt `i` tăng dần, mỗi list tăng dần, đúng thứ tự `i1 < i2 < ... < im` của đề.
- **Kiểm tra đúng:** "mọi hiệu liên tiếp bằng nhau" ⇔ mọi hiệu bằng hiệu đầu tiên `gap = idx[1] - idx[0]`. `zip(idx, idx[1:])` liệt kê **đúng và đủ** các cặp liên tiếp.
- **Đếm giá trị khác nhau:** mỗi key của dict là một giá trị khác nhau, được xét đúng một lần.
- **Cách 4:** giữ **bất biến**: sau khi xử lý tới `i`, `ok` của `x` là `True` ⇔ mọi hiệu liên tiếp giữa các vị trí `<= i` của `x` đều bằng `gap`. Khi một hiệu lệch, `ok` thành `False` mãi mãi — đúng vì thêm vị trí mới không thể "sửa" một hiệu đã lệch.

# 11. Độ phức tạp

Gọi `n = len(nums)`, `d` = số giá trị khác nhau.

| Cách | Thời gian | Bộ nhớ phụ |
|---|---|---|
| 1. Brute Force | `O(n · d)` (tới `O(n²)`) | `O(n)` |
| 2. Gom nhóm | `O(n)` trung bình | `O(n)` (tổng độ dài các list = `n`) |
| 3. Mẹo `range` | có thể lớn hơn `O(n)` nếu không kiểm tra độ dài | `O(n)` |
| 4. Một lượt, 4 thông tin | `O(n)` trung bình | `O(d)` |

"Trung bình" vì thao tác `dict` là `O(1)` **trung bình** (bảng băm), không tuyệt đối trong mọi trường hợp.

# 12. Những lỗi thường gặp

## Lỗi 1 — Quên điều kiện "ít nhất 3 lần"

Với 2 lần xuất hiện, chỉ có **một** hiệu → luôn "đều". Nếu quên điều kiện `>= 3`, ví dụ 1 sẽ đếm cả `8` → ra 3 thay vì 2.

## Lỗi 2 — Chỉ so khoảng cách đầu và cuối

"`last - first == (count - 1) * gap`" **không đủ**. Ví dụ vị trí `0, 2, 3, 6`: `gap = 2`, `(4 - 1) * 2 = 6 = 6 - 0`, nhưng hiệu là `2, 1, 3` — không đều!

## Lỗi 3 — Đếm số **phần tử** thay vì số **giá trị khác nhau**

`[8, 8, 8, 8]` cho đáp án `1`, không phải `4`.

## Lỗi 4 — Cập nhật `last` trước khi tính hiệu (cách 4)

Gán `s[0] = i` trước rồi mới tính `g = i - s[0]` → `g` luôn bằng 0.

In [12]:
def wrong_endpoint_check(nums):
    groups = defaultdict(list)
    for i, x in enumerate(nums):
        groups[x].append(i)
    count = 0
    for idx in groups.values():
        if len(idx) >= 3:
            gap = idx[1] - idx[0]
            if idx[-1] - idx[0] == (len(idx) - 1) * gap:      # SAI: không đủ
                count += 1
    return count

def wrong_no_min3(nums):
    groups = defaultdict(list)
    for i, x in enumerate(nums):
        groups[x].append(i)
    return sum(1 for idx in groups.values() if len({b - a for a, b in zip(idx, idx[1:])}) <= 1)

nums = [7, 0, 7, 7, 0, 0, 7]     # 7 ở vị trí 0, 2, 3, 6
print("Lỗi 2 | sai:", wrong_endpoint_check(nums), "| đúng:", sol.countSpecialIntegers(nums))
nums = [1, 8, 1, 5, 1, 5, 8, 5]
print("Lỗi 1 | sai:", wrong_no_min3(nums), "| đúng:", sol.countSpecialIntegers(nums))

Lỗi 2 | sai: 1 | đúng: 0
Lỗi 1 | sai: 3 | đúng: 2


# 13. Mẹo nhận diện pattern

- Đề nói về "**các lần xuất hiện của mỗi giá trị**" (vị trí, khoảng cách, tần suất) → **Hash Map: giá trị → thông tin**.
- Cần toàn bộ vị trí → `defaultdict(list)`; chỉ cần đếm → `Counter` / `defaultdict(int)`; chỉ cần vài thống kê → lưu **trạng thái nhỏ** (`last`, `first`, `count`...).
- "Các phần tử liên tiếp" trong một dãy → `zip(a, a[1:])`.
- Kiểm tra "mọi thứ bằng nhau" → `len(set(...)) == 1` hoặc `all(x == first for x in ...)`.

# 14. Biến thể — Cách đều với khoảng cách cho trước `k`

Đếm giá trị xuất hiện `>= 3` lần và mọi hiệu liên tiếp **đúng bằng `k`**.

In [13]:
def count_spacing_exactly_k(nums, k):
    groups = defaultdict(list)
    for i, x in enumerate(nums):
        groups[x].append(i)
    return sum(1 for idx in groups.values()
               if len(idx) >= 3 and all(b - a == k for a, b in zip(idx, idx[1:])))

print(count_spacing_exactly_k([1, 8, 1, 5, 1, 5, 8, 5], 2))   # 2
print(count_spacing_exactly_k([1, 8, 1, 5, 1, 5, 8, 5], 1))   # 0
print(count_spacing_exactly_k([8, 8, 8, 8], 1))               # 1

2
0
1


# 15. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Với `nums = [2, 3, 2, 3, 2, 4, 3, 4, 4]`:

1. Gom vị trí của từng giá trị.
2. Tính các hiệu và kết luận giá trị nào special.

## Bài 2 — Dự đoán output

```python
list(zip([1, 4, 9, 16], [4, 9, 16]))
{b - a for a, b in zip([0, 5, 10, 15], [5, 10, 15])}
list(range(2, 12, 3))
```

In [14]:
print(list(zip([1, 4, 9, 16], [4, 9, 16])))
print({b - a for a, b in zip([0, 5, 10, 15], [5, 10, 15])})
print(list(range(2, 12, 3)))

[(1, 4), (4, 9), (9, 16)]
{5}
[2, 5, 8, 11]


## Bài 3 — Điền chỗ trống: gom nhóm

In [15]:
def exercise_3_group_positions(nums):
    # TODO: trả về dict: giá trị -> list vị trí (tăng dần)
    # Gợi ý: defaultdict(list), enumerate
    raise NotImplementedError("Hãy tự hoàn thiện trước khi xem đáp án.")

# assert exercise_3_group_positions([5, 1, 5]) == {5: [0, 2], 1: [1]}

## Bài 4 — Tự viết cách 4 (một lượt, trạng thái nhỏ)

In [16]:
def exercise_4_one_pass(nums):
    # TODO: state[x] = [last, gap, count, ok]
    raise NotImplementedError

## Bài 5 — Biến thể: "gần như cách đều"

Một giá trị là **gần special** nếu xuất hiện `>= 3` lần và các hiệu liên tiếp chỉ nhận **nhiều nhất 2 giá trị khác nhau**. Đếm số giá trị gần special.

Ví dụ: `[8, 6, 6, 8, 8]` → `8` có hiệu `3, 1` (2 giá trị) → gần special → đáp án `1`.

In [17]:
def exercise_5_almost_special(nums):
    # TODO
    raise NotImplementedError

## Bài 6 — Mini challenge: trả lời sau mỗi phần tử (online)

Trả về list `ans` với `ans[i]` = số giá trị special nếu chỉ xét **tiền tố** `nums[0..i]`. Yêu cầu `O(n)` tổng cộng.

Ví dụ: `[8, 8, 8, 1, 8]` → `[0, 0, 1, 1, 0]` (ở `i = 4`, `8` ở `0,1,2,4` — hỏng).

In [18]:
def exercise_6_online(nums):
    # TODO: cách 4 + biến đếm current, cập nhật khi trạng thái special của x thay đổi
    raise NotImplementedError

## Gợi ý theo mức độ

**Bài 4**
- Mức 1: lần đầu gặp `x`: `[i, None, 1, True]`.
- Mức 2: lần sau: tính `g = i - last` **trước**, rồi mới cập nhật `last`.
- Mức 3: `gap is None` → đặt `gap = g`; ngược lại `g != gap` → `ok = False`.

**Bài 5**
- Mức 1: giống cách 2, chỉ đổi điều kiện `len(gaps) == 1` thành `len(gaps) <= 2`.

**Bài 6**
- Mức 1: "special" của `x` là một hàm của `(count, ok)`: `count >= 3 and ok`.
- Mức 2: trước khi cập nhật trạng thái của `x`, ghi nhớ `was = special(x)`; sau cập nhật, `now = special(x)`; `current += now - was`.
- Mức 3: chỉ trạng thái của `x = nums[i]` thay đổi ở bước `i`.

# 16. Đáp án tham khảo

**Bài 1:** `nums = [2, 3, 2, 3, 2, 4, 3, 4, 4]`

| giá trị | vị trí | hiệu | special? |
|---|---|---|---|
| 2 | `0, 2, 4` | `2, 2` | ✔ |
| 3 | `1, 3, 6` | `2, 3` | ✘ |
| 4 | `5, 7, 8` | `2, 1` | ✘ |

Đáp án `1`.

**Bài 2:** `[(1, 4), (4, 9), (9, 16)]`, `{5}`, `[2, 5, 8, 11]`.

In [19]:
assert sol.countSpecialIntegers([2, 3, 2, 3, 2, 4, 3, 4, 4]) == 1
print("Bảng đáp án bài 1 đã được kiểm chứng ✔")

Bảng đáp án bài 1 đã được kiểm chứng ✔


In [20]:
def answer_3_group_positions(nums):
    groups = defaultdict(list)
    for i, x in enumerate(nums):
        groups[x].append(i)
    return dict(groups)

assert answer_3_group_positions([5, 1, 5]) == {5: [0, 2], 1: [1]}
print("Đáp án bài 3 PASS:", answer_3_group_positions([1, 8, 1, 5, 1, 5, 8, 5]))

Đáp án bài 3 PASS: {1: [0, 2, 4], 8: [1, 6], 5: [3, 5, 7]}


In [21]:
def answer_4_one_pass(nums):
    state = {}
    for i, x in enumerate(nums):
        if x not in state:
            state[x] = [i, None, 1, True]
            continue
        s = state[x]
        g = i - s[0]
        if s[1] is None:
            s[1] = g
        elif g != s[1]:
            s[3] = False
        s[0] = i
        s[2] += 1
    return sum(1 for last, gap, cnt, ok in state.values() if cnt >= 3 and ok)

random.seed(4)
for _ in range(1000):
    nums = [random.randint(1, 3) for _ in range(random.randint(3, 12))]
    assert answer_4_one_pass(nums) == count_special_bruteforce(nums)
print("Đáp án bài 4 PASS")

Đáp án bài 4 PASS


In [22]:
def answer_5_almost_special(nums):
    groups = defaultdict(list)
    for i, x in enumerate(nums):
        groups[x].append(i)
    count = 0
    for idx in groups.values():
        if len(idx) >= 3 and len({b - a for a, b in zip(idx, idx[1:])}) <= 2:
            count += 1
    return count

assert answer_5_almost_special([8, 6, 6, 8, 8]) == 1
assert answer_5_almost_special([1, 8, 1, 5, 1, 5, 8, 5]) == 2
assert answer_5_almost_special([7, 0, 7, 7, 0, 0, 7]) == 1     # 7: hiệu 2,1,3 (3 giá trị) ✘ ; 0: 1,4,5 -> hiệu 3,1 ✔
print("Đáp án bài 5 PASS")

Đáp án bài 5 PASS


In [23]:
def answer_6_online(nums):
    state = {}
    current = 0
    ans = []
    for i, x in enumerate(nums):
        if x not in state:
            state[x] = [i, None, 1, True]
        else:
            s = state[x]
            was = s[2] >= 3 and s[3]
            g = i - s[0]
            if s[1] is None:
                s[1] = g
            elif g != s[1]:
                s[3] = False
            s[0] = i
            s[2] += 1
            now = s[2] >= 3 and s[3]
            current += int(now) - int(was)
        ans.append(current)
    return ans

assert answer_6_online([8, 8, 8, 1, 8]) == [0, 0, 1, 1, 0]
random.seed(6)
for _ in range(500):
    nums = [random.randint(1, 3) for _ in range(random.randint(3, 10))]
    assert answer_6_online(nums) == [count_special_bruteforce(nums[:i + 1]) for i in range(len(nums))]
print("Đáp án bài 6 PASS:", answer_6_online([1, 8, 1, 5, 1, 5, 8, 5]))

Đáp án bài 6 PASS: [0, 0, 0, 0, 1, 1, 1, 2]


# 17. Tổng kết

## Kiến thức Python

```python
from collections import defaultdict
groups = defaultdict(list); groups[x].append(i)
for x, idx in groups.items():
zip(idx, idx[1:])                 # cặp liên tiếp
{b - a for a, b in ...}           # set comprehension
all(...), len(set(...)) == 1
x is None                         # kiểm tra None
count += True                     # bool là số nguyên 0/1
```

## Kỹ thuật thuật toán

1. **Gom nhóm vị trí theo giá trị** bằng Hash Map, một lượt duyệt.
2. **Cách đều** ⇔ mọi hiệu liên tiếp bằng hiệu đầu tiên.
3. Tiết kiệm bộ nhớ: chỉ lưu **trạng thái nhỏ** (`last`, `gap`, `count`, `ok`).
4. Cẩn thận "mẹo gọn" có thể tốn tài nguyên ở trường hợp xấu.

## Công thức nhớ nhanh

> `special(x)` ⇔ `len(idx) >= 3` **và** `all(b - a == idx[1] - idx[0] for a, b in zip(idx, idx[1:]))`.

## Độ phức tạp

```text
Time:  O(n) trung bình
Space: O(n)  (cách 4: O(số giá trị khác nhau))
```

## Checklist trước khi nộp

- [ ] Có điều kiện `>= 3` lần?
- [ ] Kiểm tra **mọi** hiệu liên tiếp (không chỉ đầu-cuối)?
- [ ] Đếm số **giá trị khác nhau**?
- [ ] Không quét lại cả mảng cho từng giá trị?

# 18. Phiên bản cuối cùng nên nhớ

```python
from collections import defaultdict

class Solution:
    def countSpecialIntegers(self, nums: list[int]) -> int:
        groups = defaultdict(list)
        for i, x in enumerate(nums):
            groups[x].append(i)

        count = 0
        for idx in groups.values():
            if len(idx) >= 3:
                gap = idx[1] - idx[0]
                if all(b - a == gap for a, b in zip(idx, idx[1:])):
                    count += 1
        return count
```

**Gom nhóm bằng Hash Map, rồi kiểm tra từng nhóm** — khuôn mẫu này giải được rất nhiều bài về "các lần xuất hiện của một giá trị".